# Welcome to Aminoacid ! 🧬🎉

Welcome to **aminoacid**, a fun game designed to easily learn and test your skills for the different aminoacids.

The main functionalities of this python package will be explained in the following, but first...

🔨 **Let's import everything !** 

To import all the files needed to run the code properly, run the following code:

In [ ]:
import streamlit as st
from streamlit_ketcher import st_ketcher 
from rdkit import Chem 
from rdkit.Chem import Draw 
from rdkit.Chem import rdMolDescriptors 
import random 
import base64
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..")))
from data.aminoacidlist import amino_acids
from data.aminoacidlist import aa_stereo

Now that we are all set up, let's dive in ! 🏊‍♂️ 

# Introduction 📚

Amino acids, which make up proteins, are essential for the proper functioning of life. They help repair tissues, absorb nutrients and participate in the transport of neurotransmitters and so on.
There are 20 amino acids, all composed of a carboxyl group, an amino group, a hydrogen and a side chain that differentiates them. 

Having said all that, it is important to know about them and their structures, which give us an idea of their chemical properties.
It can be difficult to learn these 20 different structures. That's why we've decided to create a platform where you can learn them simply using flashcards in a grid layout and test your knowledge by drawing and naming them. 

# Material and methods 

## 1. Learn amino acids 📖

**Goal**: Learn the structure of amino acids

Let's learn the amino acids !

### 1.1 Informations about amino acids

An explication and an image of the structure of amino acids are displayed thanks to streamlit API functions. With st.caption text in small font is shown. st.write is used to displays text and st.image displays an image from a file.  

In [ ]:
st.caption("What is the structure of a amino acid?") 
st.write("An amino acid contains both amino and carboxylic acid functional group, a carbon alpha and a side chain which is variable."
         " In nature you can only find the L-configuration of amino acids, therefore they will be drawn in this configuration."
        )
img= "../../assets/aastruct.jpeg"
st.image(img, caption="Amino acid structure", use_container_width=True)

### 1.2 The visualisation of the amino acids 

A grid layout of buttons is displayed. Once clicked the button shows or hides the image of the corresponding amino aicd.

<img src="../assets/learn_grid.png" width="600"> 

To keep the image once the button is clicked, it is necessary to have it in a specific session as streamlit reruns everything each time the user interacts with the app. 
The state "visible" is intialized with a key for every element in the dictionnary aa_stereo, initally all set to False to be hidden. 

The number of columns per row is set at 4. And the dictionnary aa_stereo is converted into a list of tuples (name, smiles) for iteration. 

The loop enables the creation of a grid by inserting column containers in a row.
Each row is filled with buttons and it's image. For every amino acid a button is shown with its name. When the button is clicked the visiblity is toggled. If the session state visible[name] is true, the smiles is converted to a molecule and the molecule is drawn as an image using RDkit. This image is then displayed with st.image(). 

In [ ]:
for row_start in range(0, len(aa_items), columns_per_row):
    cols = st.columns(columns_per_row)
    row_items = aa_items[row_start: row_start + columns_per_row]
    for i in range(len(row_items)):
        col = cols[i]
        name, smiles = row_items[i]
        with col:
            if st.button(name, key=f"aa_{name}"):
                st.session_state.visible[name] = not st.session_state.visible[name]

            if st.session_state.visible[name]:
                mol = Chem.MolFromSmiles(smiles)
                img = Draw.MolToImage(mol, size=(400,400))
                st.image(img)

## 2. Game 🎮

Now that you master your amino acids, let's test your knowledge with an interactive game ! 

### 2.1 Draw amino acids 🖍

**Goal:** draw the amino acid asked for and receive an instantaneous feedback.
Keep track of your progression thanks to the progression bar, and retry your mistakes only until you are officially an amino acid expert 🎉

#### Initialization and state management

As Streamlit reruns the whole code everytime the user interacts with the app, it is of crucial importance to make sure that the actual state is maintained throughout the session. As an example, the user's score should not undergo a reset everytime a button is pressed.
This is why are first defined all keys necessary to a fluid functioning of the drawing part of the quiz through *st.session_state* command.

In [ ]:
if "retry_mode" not in st.session_state:
    st.session_state.retry_mode = False
if "incorrect_answers" not in st.session_state:
    st.session_state.incorrect_answers = []
if "round_order" not in st.session_state:
    st.session_state.round_order = random.sample(
        st.session_state.incorrect_answers if st.session_state.retry_mode else list(amino_acids.keys()),
        len(st.session_state.incorrect_answers) if st.session_state.retry_mode else len(amino_acids)
    )
if "ketcher_key" not in st.session_state:
    st.session_state.ketcher_key = 0
if "current_index" not in st.session_state:
    st.session_state.current_index = 0
if "score" not in st.session_state:
    st.session_state.score = 0
if "answered" not in st.session_state:
    st.session_state.answered = False
if "show_score_clicked" not in st.session_state:
    st.session_state.show_score_clicked = False

The keys initialized are:
- *retry_mode*: to differentiate the first round from retrying incorrect answers.
- *incorrect_answers*: to store all incorrect answers for future retry (in a list).
- *round_order*: to define the sequence of questions **randomly**. If in retrymode, only the incorrect answers are taken into account; otherwise all 20 amino acids are included.
- *ketcher_key*: to refresh the Ketcher interface where structures are drawn between each question.
- *current_index*: to keep track of the current question position. It is incremented everytime the next question is reached.
- *score*: to count the number of correct answers.
- *answered* and *show_score_clicked*: to control single answer and differentiate the quiz interface from the result screen.

#### Question and progress bar

Both question and progress bar are here to guide the user through the quiz. 

The current amino acid is displayed with command *markdown* and updated at each new question according to the *round_order* sequence.

As for the progress bar, it appears above the drawing interface. Progress is calculated as ratio of the current question index over total number of questions.

In [ ]:
progress = (st.session_state.current_index + 1) / total_questions
st.progress(progress, text=f"Progress : {st.session_state.current_index + 1} / {total_questions}")

#### Ketcher drawing interface

The drawing interface is central to this game mode. The Ketcher chemical structure editor is used for its high performance and adaptability through *st_ketcher* command. When the user is satisfied with his answer and clicks on "Apply", is generated the SMILES string corresponding to the structure drawn. 

The function *are_equivalent* checks if the user's structure is equivalent to the correct one. In order to do so, the SMILES strings are converted to non-ambiguous InChI representation (RDKit library). Indeed, different SMILES versions exist for the same compound, while a unique canonical InChI exists for a given compound.

In [ ]:
def are_equivalent(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        return False
    return Chem.MolToInchi(mol1) == Chem.MolToInchi(mol2)

A success message is displayed if the structures are equivalent.
However if the molecules don't match, an error message and an image of the correct structure are shown.

In [ ]:
st.markdown(f"The answer is : ")
mol = Chem.MolFromSmiles(target_smiles)
img = Draw.MolToImage(mol, size=(300, 300))
st.image(img, caption=f"Structure of {current_target}", use_container_width=False)

#### Scoring system

The main issue here is to make sure that the answer is considered correct only at the first attempt.

On the one hand, the score is incremented only if key *answered* is *False*:

In [ ]:
if are_equivalent(ketcher_smiles, target_smiles):
    if not st.session_state.answered:
        st.session_state.score += 1
        st.session_state.answered = True

On the other hand, it is made sure that questions can be answered only once or a message indicating to click onto "Next" will appear:

In [ ]:
if not st.session_state.answered:
    ketcher_smiles = st_ketcher(height=600, key=f"ketcher_{st.session_state.ketcher_key}")
else:
    st.info("You already answered! Click **Next** to continue.")
    ketcher_smiles = None

At the end of the round, the score is displayed after clicking on the "Show Score" button. According to the score percentage is proposed a different message: 

In [ ]:
if percent_score == 100:
    st.balloons()
    st.markdown("🎉 You're an amino acid expert !")
elif percent_score >= 75:
    st.markdown("Great job 👏 : You know your stuff !")
elif percent_score >= 50:
    st.markdown("🧪 Keep practicing and you'll get there.")
else:
    st.markdown("📚 It's time to hit the books, don't give up !")

#### Retry mistakes option

When the round has been finished, it is possible to retry only the amino acids drawn incorrectly. This is made possible by the *incorrect_answers* key defined previously. It is a list that stores all amino acids drawn wrong. 

If this option is chosen, the retry round will work exactly the same way as the first round, only with less amino acids. All keys are reset and the *retry_mode* is activated. To be noted that the cycle is not limited, meaning that the user can retry until every answer is right.

In [ ]:
if st.button("Retry Mistakes"):
    st.session_state.retry_mode = True
    st.session_state.round_order = random.sample(
        st.session_state.incorrect_answers, len(st.session_state.incorrect_answers)
    )
    st.session_state.current_index = 0
    st.session_state.score = 0
    st.session_state.answered = False
    st.session_state.show_score_clicked = False
    st.session_state.incorrect_answers = []
    st.rerun()

### 2.2 Name amino acids

This section allows you to practise naming the different amino acids by looking at their structure. It was created using the **name_quizz()** function. 

As for the other sections, the representations of the amino acids are obtained using smiles and grouped together in a dictionnary, **amino_acids**. It is imported from a file using the following commands:

In [ ]:
from data.aminoacidlist import amino_acids

When you play the game, after entering the name of the amino acid, press the ‘Check answer’ button and you'll find out whether the answer given is correct or not. It is possible thanks to the following commands:

In [ ]:
if user_guess.lower() == correct_name.lower():

So as not to have any problems with the way the user writes the name (upper case, lower case), the **.lower()** function was used.

If the answer is incorrect, the correct answer will be given thanks to this line of code:

In [ ]:
st.error(f"❌ Nope! The correct answer was: **{correct_name}**")

# Limitations

When the user is in light mode or dark mode on the computer, the interface and color of the text changes with the mode. Indeed, in the dark mode the text becomes white and it is less readable. Therefore, it is best to be in the light mode. 